# 📊 MLP Model Training - Hierarchical Demand Forecasting

## 🔍 Въведение
Този notebook е посветен на обучението на Multi-Layer Perceptron (MLP) модел за прогнозиране на продажби. Моделът използва обработените данни от DataProcessing.ipynb и следва архитектурата, специално разработена за таблични данни с lag features, rolling statistics и календарни характеристики.

---

## 🎯 Цел
Изграждане и обучение на MLP модел за regression задача (прогнозиране на продажби 1 ден напред), който:
✅ Използва обработените данни от DataProcessing pipeline
✅ Следва оптимална архитектура за таблични данни
✅ Включва Batch Normalization и Dropout за стабилност и regularization
✅ Имплементира early stopping за предотвратяване на overfitting
✅ Използва подходящи метрики за regression (MSE, MAE, RMSE)

---

## 📋 План на notebook-а

| Секция | Описание |
|--------|----------|
| **0. Setup** | Импортиране на библиотеки, настройка на пътища |
| **1. Load Processed Data** | Зареждане на X_train_scaled, y_train и т.н. |
| **2. Model Architecture** | Дефиниране на MLP архитектурата |
| **3. Compile Model** | Конфигуриране на optimizer, loss и metrics |
| **4. Training** | Обучение с validation и early stopping |
| **5. Evaluation** | Оценка на модела върху test set |
| **6. Save Model** | Запазване на обучения модел |

---


## 0. Setup


In [10]:
import torch
import os
os.environ["KERAS_BACKEND"] = "torch"

from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import json

from sklearn.metrics import mean_squared_error, mean_absolute_error
import keras
from keras import layers, callbacks

# Проектови пътища
# Определяне на root директорията (работи независимо от текущата директория)
if Path.cwd().name == "models":
    PROJ_ROOT = Path.cwd().resolve().parent
else:
    PROJ_ROOT = Path.cwd().resolve()

DATA_PROCESSED = PROJ_ROOT / "data" 
MODELS_DIR = PROJ_ROOT / "models"
DATA_DIR = PROJ_ROOT / "data"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Проверка за GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Seed за възпроизводимост
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

print(f"\n✅ Setup завършен")
print(f"Keras backend: {keras.config.backend()}")


CUDA available: False

✅ Setup завършен
Keras backend: torch


## 1. Load Processed Data

Зареждаме обработените данни от DataProcessing pipeline.

**Как да заредите данните:**

1. **Препоръчителен метод:** Изпълнете DataProcessing.ipynb (секции 0-9) - той автоматично запазва данните в `data/processed/processed_data.pkl`
2. Клетката по-долу автоматично зарежда данните от файла (по-бързо и надеждно)
3. **Алтернативен метод:** Ако файлът не съществува, опитва се да използва `%run` magic

След зареждане ще имате в паметта:
- `X_train_scaled`, `y_train`
- `X_val_scaled`, `y_val`  
- `X_test_scaled`, `y_test`


In [ ]:
processed_data_path = DATA_PROCESSED / "processed_data.pkl"

if processed_data_path.exists():
    print(f"📂 Зареждане на данни от файл: {processed_data_path}")
    with open(processed_data_path, "rb") as f:
        data = pickle.load(f)
    
    X_train_scaled = data["X_train_scaled"]
    y_train = data["y_train"]
    X_val_scaled = data["X_val_scaled"]
    y_val = data["y_val"]
    X_test_scaled = data["X_test_scaled"]
    y_test = data["y_test"]
    
    print("✅ Данните са заредени от файл!")
else:
    print(f"⚠️  Файлът {processed_data_path} не съществува")
    print("\n💡 Решение:")
    print("   1. Изпълнете DataProcessing.ipynb (секции 0-9) за да създадете файла")
    print("   2. Или използвайте %run magic по-долу (ако е наличен)")
    
    # Алтернативен метод: %run magic
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython is not None:
            print("\n🔄 Опит за зареждане чрез %run magic...")
            ipython.run_line_magic('run', '../data/DataProcessing.ipynb')
            print("✅ Данните са заредени чрез %run magic")
        else:
            raise NameError("Not running in IPython environment")
    except (NameError, ModuleNotFoundError, AttributeError) as e:
        print(f"\n❌ %run magic не е наличен: {e}")
        raise ValueError("Няма налични данни! Изпълнете DataProcessing.ipynb първо.")


📂 Зареждане на данни от файл: /Users/filipapopova/source/repos/2526-12b-feedforwardneuralnetwork-hierarchical-demand-forecasting/data/processed_data.pkl
✅ Данните са заредени от файл!


In [12]:
# Проверка на заредените данни
print("📊 Проверка на данните...\n")

try:
    # Проверяваме дали данните са в паметта
    X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test
    
    print("✅ Данните са успешно заредени!\n")
    
    # Размери на данните
    print(f"📊 Данни за обучение:")
    print(f"   X_train_scaled: {X_train_scaled.shape}")
    print(f"   y_train: {y_train.shape}")
    print(f"\n📊 Данни за валидация:")
    print(f"   X_val_scaled: {X_val_scaled.shape}")
    print(f"   y_val: {y_val.shape}")
    print(f"\n📊 Данни за тестване:")
    print(f"   X_test_scaled: {X_test_scaled.shape}")
    print(f"   y_test: {y_test.shape}")
    
except NameError as e:
    print(f"❌ Данните не са заредени: {e}")
    print("\n💡 Решение:")
    print("   1. Изпълнете DataProcessing.ipynb (секции 0-9) за да създадете processed_data.pkl")
    print("   2. Или използвайте %run magic в клетката по-горе")
    raise


📊 Проверка на данните...

✅ Данните са успешно заредени!

📊 Данни за обучение:
   X_train_scaled: (755970, 22)
   y_train: (755970,)

📊 Данни за валидация:
   X_val_scaled: (162155, 22)
   y_val: (162155,)

📊 Данни за тестване:
   X_test_scaled: (162155, 22)
   y_test: (162155,)


In [13]:
# Подготовка на данните за обучение

# Брой на features
n_features = X_train_scaled.shape[1]
print(f"✅ Брой features: {n_features}")

# Конвертиране към numpy arrays (ако са pandas DataFrames)
if isinstance(X_train_scaled, pd.DataFrame):
    X_train_scaled = X_train_scaled.values
    X_val_scaled = X_val_scaled.values
    X_test_scaled = X_test_scaled.values

if isinstance(y_train, pd.Series):
    y_train = y_train.values
    y_val = y_val.values
    y_test = y_test.values

# Конвертиране към float32 за по-добра производителност
X_train_scaled = X_train_scaled.astype("float32")
X_val_scaled = X_val_scaled.astype("float32")
X_test_scaled = X_test_scaled.astype("float32")
y_train = y_train.astype("float32")
y_val = y_val.astype("float32")
y_test = y_test.astype("float32")

print(f"\n✅ Данните са готови за обучение (float32)")
print(f"   Train: {len(X_train_scaled):,} редове")
print(f"   Val:   {len(X_val_scaled):,} редове")
print(f"   Test:  {len(X_test_scaled):,} редове")


✅ Брой features: 22

✅ Данните са готови за обучение (float32)
   Train: 755,970 редове
   Val:   162,155 редове
   Test:  162,155 редове
